# LoRa SNR prediction: demo
Run with **Kernel → Restart & Run All**. Uses only the saved predictions (no model training, about 2 minutes). Put this notebook inside the `madeira_snr` folder.

In [ ]:
import os
print('Working folder:', os.getcwd())
assert os.path.exists('07_analysis.py'), 'Open this notebook from inside the madeira_snr folder'

## 1. Integrity checks

In [ ]:
!python 06_check.py

In [ ]:
!python tools/verify_results.py

## 2. Recompute analysis, tables and figures from saved predictions

In [ ]:
!python 07_analysis.py > logs/07_demo.log && echo 07_analysis done
!python 05_summarise.py > logs/05_demo.log && echo 05_summarise done

## 3. Main results

In [ ]:
import json, pandas as pd
pd.set_option('display.width', 200)
print(json.load(open('results/calibration/buffer_choice.json')))
pd.read_csv('results/summary/table1_rmse_by_scheme.csv')

In [ ]:
h = json.load(open('results/analysis/h1_verdict.json'))
rows = [dict(direction=d, **{k: h[d]['block500_all'][k] for k in ['delta_loc_Bstar','ci95','classification','delta_loc_B0_same_packets','change_Bstar_minus_B0','change_ci95']}) for d in ['Uplink','Downlink']]
pd.DataFrame(rows)

In [ ]:
pd.read_csv('results/summary/table2_paired_gains.csv')

In [ ]:
pd.read_csv('results/summary/table5_buffer_series.csv')

## 4. Figures

In [ ]:
from IPython.display import Image, display
for f in ['fig1_rmse_by_scheme','fig2_gain_decomposition','fig3_buffer_series','fig4_buffer_series_B1000_supported']:
    display(Image(f'figures/{f}.png'))

## 5. Channel modelling (path-loss reproduction of Azevedo & Mendonça 2026, Table 3)

In [ ]:
!python 09_pathloss_reproduction.py > logs/09_demo.log
pd.read_csv('results/pathloss/fi_fit_vs_paper_table3.csv').query("direction=='Uplink' and distance=='d_3d'")

## 6. Live model training (demo subset)
Trains the models **from scratch** on 12 test blocks (uplink), with no buffer and with the 500 m buffer, using the same code and fixed hyperparameters as the full pipeline. About 1 minute.

**This is a demonstration on a subset. The reported results come from all 126 blocks (Section 3).**

In [ ]:
import json, time, numpy as np, pandas as pd
from models import load_packets, fit_predict_primary, FEAT_DIST, FEAT_LOC
pk = load_packets()
lf = pd.read_csv('splits/location_folds.csv', dtype={'location_id': str})
pk = pk.merge(lf[['location_id', 'block250']], on='location_id')
ex = pd.read_csv('splits/lobo_excluded.csv', dtype={'location_id': str, 'block': str})
hp = json.load(open('results/eval/hp_fixed.json'))['Uplink']
D = pk[pk.direction == 'Uplink'].reset_index(drop=True)
blocks = np.random.default_rng(0).choice(sorted(D.block250.unique()), 12, replace=False)
print('Features, distance-only RF :', FEAT_DIST)
print('Features, location-aware RF:', FEAT_LOC)
print('Fixed hyperparameters      :', hp)

In [ ]:
rows = []; t0 = time.time()
for b in (0, 500):
    preds, ys = {m: [] for m in ['logdist', 'rf_dist', 'rf_loc']}, []
    for blk in blocks:
        te = D[D.block250 == blk]
        drop = set(ex[(ex.grid == 'o0') & (ex.buffer_m == b) & (ex.block == blk)].location_id)
        tr = D[(D.block250 != blk) & ~D.location_id.isin(drop)].reset_index(drop=True)
        assert not set(te.location_id) & set(tr.location_id)          # no test location in training
        p, _ = fit_predict_primary(tr, te, hp)                          # trains log-distance, RF-dist, RF-loc
        for m in preds: preds[m].append(p[m])
        ys.append(te.snr_db.values)
    y = np.concatenate(ys)
    r = {m: float(np.sqrt(np.mean((np.concatenate(v) - y) ** 2))) for m, v in preds.items()}
    rows.append(dict(buffer_m=b, test_packets=len(y), RMSE_logdist=r['logdist'], RMSE_rf_dist=r['rf_dist'],
                     RMSE_rf_loc=r['rf_loc'], delta_loc=r['rf_loc'] - r['rf_dist']))
    print(f'buffer {b} m done ({time.time() - t0:.0f} s)')
pd.DataFrame(rows).round(2)